In [ ]:
import pandas as pd
import pycountry
from rapidfuzz import process
from pathlib import Path
import csv

#output path
path = r"C:\Users\Willi\Desktop\Data Analytic Projects\Population analysis\cleaned datasets"
name = r"\worldbank_clean_countries_dataset.csv"

#reads the files
df = pd.read_csv(r"C:\Users\Willi\Desktop\Data Analytic Projects\Population analysis\original datasets\wolrdbank_population.csv")

#selected columns only
years = [str(year) for year in range (1960, 2025)]
df = df[["Country Name","Country Code"]+ years]

# Hardcoded Country names standarization plus validations
valid_countries = {c.name for c in pycountry.countries}

hardcoded_country = {
    "Venezuela, RB": "Venezuela",
    "Iran, Islamic Rep.": "Iran",
    "Korea, Rep.": "South Korea",
    "Korea, Dem. People's Rep.": "North Korea",
    "Egypt, Arab Rep.": "Egypt",
    "Russian Federation": "Russia",
    "Syrian Arab Republic": "Syria",
    "Yemen, Rep.": "Yemen",
    "Viet Nam": "Vietnam",
    "Tanzania, United Republic of": "Tanzania",
    "St. Martin (French part)": "French St. Martin",
    "Sint Maarten (Dutch part)": "Dutch Sint Maarten",
    "Puerto Rico (US)": "Puerto Rico",
    "Micronesia, Fed. Sts.": "Micronesia",
    "Moldova, Republic of": "Moldova",
    "Bahamas, The": "Bahamas",
    "Virgin Islands, British": "British Virgin Islands",
    "Virgin Islands (U.S.)": "American Virgin Islands",
    "West Bank and Gaza": "Palestine",
    "Congo, Dem. Rep.": "Democratic Republic of the Congo",
    "Congo, Rep.": "Republic of the Congo",
    "Somalia, Fed. Rep.": "Somalia",
    "Slovak Republic": "Republic of Slovakia",
}

def standardized_name(name):
    if pd.isna(name):
        return None

    name = str(name).strip()

    if name in hardcoded_country:
        return hardcoded_country[name]
    try:
        return pycountry.countries.lookup(name).name
    except:
        pass
    best, score, _ = process.extractOne(name, valid_countries)

    if score >= 85:
        return best

    return name

# apply function
df["Country Name"] = df["Country Name"].apply(standardized_name)

# duplicates and inconsistencies removal
priority_codes = ["ZAF","CAF","PAK","AFG","SYR","ATG","BIH","TCA", "TTO", "ARE"]
df["_priority"] = df["Country Code"].isin(priority_codes)
df = df.sort_values(["Country Name", "_priority"], ascending=[True, False])
df = df.drop_duplicates(subset=["Country Name"], keep="first")
df = df.drop(columns="_priority").reset_index(drop=True)

# Invalid countries drop by codes
invalid_codes = [
    "CSS", "EAR", "EAS", "EAP", "TEA", "EMU", "ECS", "ECA", "TEC", "EUU", "INX",
    "HPC", "HIC", "IBD", "IBT", "IDB", "IDX", "IDA", "LTE", "LCN", "LAC", "UMC",
    "LDC", "LMY", "LIC", "LMC", "MEA", "MNA", "TMN", "MIC", "NAC", "OED", "OSS",
    "PSS", "PST", "PRE", "SST", "TSA", "SSF", "SSA", "TSS", "TLA", "CHI", "FCS",
    "AFE", "SAS", "VAT", "TWN", "AFW", "ARB", "CEB", "WLD"]

df = df[~df["Country Code"].isin(invalid_codes)]

#drop n/a values
df = df.dropna(subset=["Country Name"])

#country area external dataset merge for completeness
area_df = pd.read_csv(r"C:\Users\Willi\Desktop\Data Analytic Projects\Population analysis\cleaned datasets\countries_(land_area, continents, capital).csv")
df = df.merge(area_df, on=["Country Name","Country Code"], how="left")

#save to csv
df.to_csv(path + name, index=False,quoting=csv.QUOTE_MINIMAL)

print(f"The {name} file has been successfully saved to {path}")

print(df[["Country Name", "Country Code", "Continent", "Capital", "Land Area (km2)"]].to_string())


The \worldbank_clean_countries_dataset.csv file has been successfully saved to C:\Users\Willi\Desktop\Data Analytic Projects\Population analysis\cleaned datasets
                         Country Name Country Code      Continent                    Capital  Land Area (km2)
0                         Afghanistan          AFG           Asia                      Kabul           652230
1                             Albania          ALB         Europe                     Tirana            27400
2                             Algeria          DZA         Africa                    Algiers          2381740
3                      American Samoa          ASM        Oceania                  Pago Pago              200
4                             Andorra          AND         Europe           Andorra la Vella              470
5                              Angola          AGO         Africa                     Luanda          1246700
6                 Antigua and Barbuda          ATG  North America   